# Cross-omics correlation analysis

Pearson and Spearman feature-to-feature correlations across transcriptomics, proteomics, and metabolomics.

## 1. Imports

In [3]:
import pandas as pd
import numpy as np

from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests

## 2. Load differential-expression results and select features

In [5]:
# choose transcripts and proteins that are also in significant ones after analysis

p_sign = pd.read_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/data/limma_proteomics.csv")
p_sign

,ID,logFC,AveExpr,t,P.Value,adj.P.Val,B
0,P19037,9.389349,23.960187,40.232488,8.250000e-28,4.570000e-24,48.899841
1,Q9SKY8,3.233129,22.868083,23.558202,5.270000e-21,1.460000e-17,36.755390
2,Q38806,9.073116,23.830519,23.743431,9.930000e-20,1.840000e-16,33.674259
3,Q9LHA8,1.813143,27.132859,18.846165,2.940000e-18,4.070000e-15,31.122530
4,P42730,4.198082,24.123611,18.426947,5.500000e-18,5.080000e-15,30.548011
...,...,...,...,...,...,...,...
5539,AT3G09250.2,-0.000187,20.547195,-0.001479,9.988294e-01,9.997092e-01,-6.903918
5540,Q9FLP6,0.000097,24.410395,0.000823,9.993484e-01,9.997092e-01,-6.903919
5541,Q8VWG7,-0.000009,22.136606,-0.000064,9.999491e-01,9.999491e-01,-6.903919
5542,AT5G16700.1,NaN,21.927257,NaN,NaN,NaN,NaN


In [12]:
p_selected = p_sign[p_sign['adj.P.Val']<0.05].ID.to_list()

In [41]:
t_sign = pd.read_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/data/DESeq2_PRIMING_vs_CONTROL_general.csv")
t_selected = t_sign[(abs(t_sign.log2FoldChange)>=1.5)&(t_sign.padj<0.05)].ID.to_list()

## 3. Load normalized omics matrices

In [42]:
transcriptomics = pd.read_csv(
    "lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/data/transcriptomics_deseqnorm+VST.csv",
    index_col=0
)
#transcriptomics.index = transcriptomics.index.str.split(',').str[0].str.split('.').str[0]

proteomics = pd.read_csv(
    "lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/data/proteomics_mediannorm+log.csv",
     index_col=0
)

metabolomics = pd.read_csv(
    "lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/data/metabolomics_log.csv",
     index_col=0
)

# Convert all abundance values to numeric
transcriptomics = transcriptomics.apply(pd.to_numeric, errors="coerce")
proteomics = proteomics.apply(pd.to_numeric, errors="coerce")
metabolomics = metabolomics.apply(pd.to_numeric, errors="coerce")

## 4. Filter transcriptomics and proteomics features

In [43]:
transcriptomics = transcriptomics.loc[transcriptomics.index.isin(t_selected)]
proteomics = proteomics.loc[proteomics.index.isin(p_selected)]

## 5. Identify and align shared samples

In [44]:
common_samples = sorted(
    set(transcriptomics.columns)
    & set(proteomics.columns)
    & set(metabolomics.columns)
)

print("Number of shared samples:", len(common_samples))
print(common_samples)

# Align all datasets
transcriptomics = transcriptomics[common_samples]
proteomics = proteomics[common_samples]
metabolomics = metabolomics[common_samples]

# Check alignment
assert list(transcriptomics.columns) == list(proteomics.columns)
assert list(transcriptomics.columns) == list(metabolomics.columns)

Number of shared samples: 22
['WTcontrol1d_2', 'WTcontrol1d_4', 'WTcontrol2d_1', 'WTcontrol2d_2', 'WTcontrol2d_3', 'WTcontrol2d_4', 'WTcontrol3d_1', 'WTcontrol3d_2', 'WTcontrol3d_3', 'WTcontrol3d_4', 'WTcontrol4d_2', 'WTcontrol4d_3', 'WTcontrol4d_4', 'WTpriming2d_1', 'WTpriming2d_3', 'WTpriming3d_1', 'WTpriming3d_2', 'WTpriming3d_3', 'WTpriming3d_4', 'WTpriming4d_2', 'WTpriming4d_3', 'WTpriming4d_4']


## 6. Correlation calculation function

In [52]:

import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests


def calculate_omics_correlations(
    data1,
    data2,
    name1,
    name2,
    min_samples=5,
    min_std=None
):
    """
    Calculate pairwise Pearson and Spearman correlations
    between features from two omics datasets.

    Rows: features
    Columns: samples

    FDR correction is applied separately for Pearson
    and Spearman correlations.
    """

    # Validate input
    if not isinstance(data1, pd.DataFrame) or not isinstance(data2, pd.DataFrame):
        raise TypeError("Both datasets must be pandas DataFrames.")

    if not isinstance(min_samples, int) or min_samples < 3:
        raise ValueError("min_samples must be an integer >= 3.")

    if min_std is not None and min_std < 0:
        raise ValueError("min_std must be nonnegative.")

    if not data1.columns.is_unique:
        raise ValueError("Duplicate sample IDs in data1.")

    if not data2.columns.is_unique:
        raise ValueError("Duplicate sample IDs in data2.")

    # Numeric conversion
    data1 = data1.apply(pd.to_numeric, errors="coerce")
    data2 = data2.apply(pd.to_numeric, errors="coerce")

    # Match samples
    common_samples = data1.columns.intersection(
        data2.columns, sort=False
    )

    if len(common_samples) < min_samples:
        raise ValueError(
            f"Only {len(common_samples)} common samples; "
            f"at least {min_samples} required."
        )

    data1 = data1.loc[:, common_samples]
    data2 = data2.loc[:, common_samples]

    # Convert once for efficiency
    array1 = data1.to_numpy(dtype=float)
    array2 = data2.to_numpy(dtype=float)

    pearson_results = []
    spearman_results = []

    comparison = f"{name1}_vs_{name2}"

    for i, feature1 in enumerate(data1.index):

        values1 = array1[i]

        for j, feature2 in enumerate(data2.index):

            values2 = array2[j]

            # Pairwise complete observations
            valid = np.isfinite(values1) & np.isfinite(values2)

            n = int(valid.sum())

            if n < min_samples:
                continue

            x = values1[valid]
            y = values2[valid]

            # Constant features
            if np.ptp(x) == 0 or np.ptp(y) == 0:
                continue

            # Optional scale-dependent variance filter
            if min_std is not None:
                if np.std(x) < min_std or np.std(y) < min_std:
                    continue

            common_info = {
                "Feature_1": feature1,
                "Feature_2": feature2,
                "Feature_1_row": i,
                "Feature_2_row": j,
                "n_samples": n,
                "Omics_Comparison": comparison
            }

            # Pearson
            r, p = pearsonr(x, y)

            if np.isfinite(r) and np.isfinite(p):
                pearson_results.append({
                    **common_info,
                    "Pearson_r": r,
                    "p_value": p
                })

            # Spearman
            rho, p_s = spearmanr(x, y)

            if np.isfinite(rho) and np.isfinite(p_s):
                spearman_results.append({
                    **common_info,
                    "Spearman_rho": rho,
                    "p_value": p_s
                })

    # Explicit schemas
    base_columns = [
        "Feature_1",
        "Feature_2",
        "Feature_1_row",
        "Feature_2_row",
        "n_samples",
        "Omics_Comparison"
    ]

    pearson_df = pd.DataFrame(
        pearson_results,
        columns=base_columns + ["Pearson_r", "p_value"]
    )

    spearman_df = pd.DataFrame(
        spearman_results,
        columns=base_columns + ["Spearman_rho", "p_value"]
    )

    # BH-FDR separately for each correlation method
    for result_df in (pearson_df, spearman_df):

        result_df["padj"] = np.nan

        if result_df.empty:
            continue

        valid_p = np.isfinite(result_df["p_value"].to_numpy())

        if valid_p.any():
            result_df.loc[valid_p, "padj"] = multipletests(
                result_df.loc[valid_p, "p_value"].to_numpy(),
                method="fdr_bh"
            )[1]

    return pearson_df, spearman_df

## 7. Check duplicate feature identifiers

In [45]:
print("Transcriptomics duplicated IDs:",
      transcriptomics.index.duplicated().sum())

print("Proteomics duplicated IDs:",
      proteomics.index.duplicated().sum())

print("Metabolomics duplicated IDs:",
      metabolomics.index.duplicated().sum())

Transcriptomics duplicated IDs: 0
Proteomics duplicated IDs: 0
Metabolomics duplicated IDs: 0


## 8. Calculate pairwise omics correlations

In [53]:
# 1. Transcriptomics vs Proteomics
tp_pearson, tp_spearman = calculate_omics_correlations(
    transcriptomics,
    proteomics,
    "Transcriptomics",
    "Proteomics"
)

# 2. Transcriptomics vs Metabolomics
tm_pearson, tm_spearman = calculate_omics_correlations(
    transcriptomics,
    metabolomics,
    "Transcriptomics",
    "Metabolomics"
)

# 3. Proteomics vs Metabolomics
pm_pearson, pm_spearman = calculate_omics_correlations(
    proteomics,
    metabolomics,
    "Proteomics",
    "Metabolomics"
)

## 9. Inspect correlation results

In [54]:
tp_spearman

,Feature_1,Feature_2,Spearman_rho,p_value,n_samples,padj,Omics_Comparison
0,AT2G34430.1,AT2G39730.2,0.207228,0.354786,22,0.645977,Transcriptomics_vs_Proteomics
1,AT2G34430.1,P19366,0.324675,0.140414,22,0.398242,Transcriptomics_vs_Proteomics
2,AT2G34430.1,Q9LD57,0.285150,0.198337,22,0.480195,Transcriptomics_vs_Proteomics
3,AT2G34430.1,Q9LHA8,-0.760587,0.000040,22,0.005921,Transcriptomics_vs_Proteomics
4,AT2G34430.1,Q9LTX9,0.489554,0.020749,22,0.139025,Transcriptomics_vs_Proteomics
...,...,...,...,...,...,...,...
59266,AT2G26020.1.1,Q8VYD2,0.153247,0.507204,21,0.764113,Transcriptomics_vs_Proteomics
59267,AT2G26020.1.1,Q9FXH1,-0.383333,0.308495,9,0.604397,Transcriptomics_vs_Proteomics
59268,AT2G26020.1.1,Q9LDA9,0.116667,0.765008,9,0.906105,Transcriptomics_vs_Proteomics
59269,AT2G26020.1.1,F4KG50,0.193676,0.387800,22,0.673744,Transcriptomics_vs_Proteomics


## 10. Filter and export significant correlations

In [55]:
def filter_significant_correlations(df, correlation_column):

    return df[
        (df["padj"] < 0.05) &
        (df[correlation_column].abs() >= 0.7)
    ].sort_values(
        by="padj"
    )


# Pearson
tp_pearson_sig = filter_significant_correlations(
    tp_pearson, "Pearson_r"
)

tm_pearson_sig = filter_significant_correlations(
    tm_pearson, "Pearson_r"
)

pm_pearson_sig = filter_significant_correlations(
    pm_pearson, "Pearson_r"
)

# Spearman
tp_spearman_sig = filter_significant_correlations(
    tp_spearman, "Spearman_rho"
)

tm_spearman_sig = filter_significant_correlations(
    tm_spearman, "Spearman_rho"
)

pm_spearman_sig = filter_significant_correlations(
    pm_spearman, "Spearman_rho"
)

# Save significant results
tp_pearson_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Transcriptomics_Proteomics_Pearson.csv", index=False)
tm_pearson_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Transcriptomics_Metabolomics_Pearson.csv", index=False)
pm_pearson_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Proteomics_Metabolomics_Pearson.csv", index=False)

tp_spearman_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Transcriptomics_Proteomics_Spearman.csv", index=False)
tm_spearman_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Transcriptomics_Metabolomics_Spearman.csv", index=False)
pm_spearman_sig.to_csv("lod-pathways-community/pilots/pairedomics_overlap/arabidopsis_heat_stress/results/Significant_Proteomics_Metabolomics_Spearman.csv", index=False)